# SmartCart Day 4 - Augment the Long Tail

Day 3's error report named the weak SKUs - usually the rare ones with too few examples. Today we first use **classical augmentation** as the reliable baseline, then leave a **cGAN/equivalent generative option** for teams that want to go further. Either way, the final judge is weak-class validation lift.

In [1]:
# 1) Runtime setup
# Install only the packages used in this notebook.
%pip install -q timm

import os

# The cross-day bundle lives in a local folder (was a Google Drive mount on Colab).
# Day 1–3 write to this same path for continuity.
BUNDLE_DIR = os.path.expanduser('~/SmartCart_bundle')



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip3.13 install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


### Embedded toolkit

This cell defines the helper functions used below. Run it once after setup.

In [2]:
from __future__ import annotations
import json
import os
import pathlib
import subprocess
import numpy as np
import pandas as pd
HERE = pathlib.Path.cwd()  # embedded in-notebook: no __file__, anchor on the working dir
GROCERY_DATASET_URL = 'https://github.com/marcusklasson/GroceryStoreDataset'

class Bundle:
    """Small local folder that carries artifacts from one day to the next."""

    def __init__(self, root: str):
        self.root = pathlib.Path(root)
        self.root.mkdir(parents=True, exist_ok=True)
        self.manifest = {'version': 1, 'class_list': [], 'artifacts': {}}

    def put_table(self, name, df: pd.DataFrame):
        df.to_csv(self.root / name, index=False)
        self._note(name)

    def get_table(self, name) -> pd.DataFrame:
        return pd.read_csv(self.root / name)

    def put_array(self, name, arr: np.ndarray):
        np.save(self.root / name, arr)
        self._note(name)

    def get_array(self, name) -> np.ndarray:
        p = self.root / name
        return np.load(p if p.suffix == '.npy' else p.with_suffix('.npy'))

    def _note(self, name):
        self.manifest['artifacts'][name] = True

    def save(self):
        (self.root / 'manifest.json').write_text(json.dumps(self.manifest, indent=2))

    def load(self):
        p = self.root / 'manifest.json'
        if p.exists():
            self.manifest = json.loads(p.read_text())
        return self

def open_bundle(drive_dir=None) -> Bundle:
    """Open the cross-day local bundle. If it is new, start with an empty manifest."""
    return Bundle(drive_dir or os.path.expanduser('~/SmartCart_bundle')).load()

def save_bundle(b: Bundle):
    b.save()
    print(f'[bundle] saved -> {b.root}')

def open_grocery_dataset():
    """Clone or reuse the real GroceryStoreDataset and return (tier, root_dir)."""
    candidates = [
        HERE / 'GroceryStoreDataset',
        HERE.parent / 'day_3' / 'GroceryStoreDataset',
        HERE.parent / 'day_2' / 'GroceryStoreDataset',
        HERE.parent / 'GroceryStoreDataset',
    ]
    for root in candidates:
        if (root / 'dataset' / 'classes.csv').exists():
            print('using cached GroceryStoreDataset:', root)
            print('using data tier: github')
            print('data root:', root)
            print('OK: using real GroceryStoreDataset images.')
            return ('github', str(root))
    root = HERE / 'GroceryStoreDataset'
    print('cloning GroceryStoreDataset from:', GROCERY_DATASET_URL)
    subprocess.run(['git', 'clone', '--depth', '1', GROCERY_DATASET_URL, str(root)], check=True)
    if not (root / 'dataset' / 'classes.csv').exists():
        raise RuntimeError('GroceryStoreDataset clone is incomplete. Check network access and rerun.')
    print('using data tier: github')
    print('data root:', root)
    print('OK: using real GroceryStoreDataset images.')
    return ('github', str(root))

def open_sg_dataset(sg_root=None):
    """SG innovation swap for open_grocery_dataset(): point at our own Singapore
    catalog (sg_data/) instead of the course-default GroceryStoreDataset. Returns
    the same (tier, root_dir) shape so downstream code doesn't need to branch."""
    root = pathlib.Path(sg_root or (HERE.parent / 'sg_data'))
    if not (root / 'labels.csv').exists():
        raise RuntimeError(f'sg_data not found at {root} (expected labels.csv). '
                            f'Check the path or run Week4/sg_data/collect_sg_groceries.py first.')
    print('using data tier: sg_data (Singapore catalog)')
    print('data root:', root)
    print('OK: using real FairPrice-sourced Singapore product images.')
    return ('sg_data', str(root))

def list_images(root, per_class=None):
    """Return a DataFrame with columns path,coarse,fine.
    Handles two layouts: GroceryStoreDataset's train/val/test split folders, and
    sg_data's flat labels.csv (path,coarse,fine) — detected by which one exists."""
    root = pathlib.Path(root)
    labels_csv = root / 'labels.csv'
    if labels_csv.exists():
        df = pd.read_csv(labels_csv)
        df['path'] = df['path'].map(lambda p: p if pathlib.Path(p).is_absolute() else str((root / p).resolve()))
    else:
        rows = []
        for split in ('train', 'val', 'test'):
            for p in (root / 'dataset' / split).rglob('*.jpg'):
                rows.append({'path': str(p), 'coarse': p.parent.parent.name, 'fine': p.parent.name})
        df = pd.DataFrame(rows)
    if per_class and len(df):
        df = df.groupby('fine', group_keys=False).head(per_class).reset_index(drop=True)
    return df

def load_backbone(name='vit_small_patch14_dinov2.lvd142m', device='cpu'):
    """Frozen DINOv2-small feature extractor."""
    import timm
    m = timm.create_model(name, pretrained=True, num_classes=0, dynamic_img_size=True).eval().to(device)
    for p in m.parameters():
        p.requires_grad_(False)
    return m

def extract_features(model, batches, device=None) -> np.ndarray:
    """Run image batches through a frozen backbone and return numpy features."""
    import torch
    if device is None:
        try:
            device = next(model.parameters()).device
        except (AttributeError, StopIteration):
            device = 'cpu'
    outs = []
    with torch.no_grad():
        for xb in batches:
            y = model(xb.to(device))
            outs.append(y.detach().cpu().numpy().astype('float32'))
    return np.concatenate(outs, 0)

class LinearHead:
    """Tiny torch linear classifier trained on frozen image features."""

    def __init__(self, in_dim, n_classes):
        import torch
        import torch.nn as nn
        self.net = nn.Linear(in_dim, n_classes)
        self.torch = torch

    def fit(self, X, y, epochs=30, lr=0.01):
        t = self.torch
        opt = t.optim.Adam(self.net.parameters(), lr=lr)
        lossf = t.nn.CrossEntropyLoss()
        Xt = t.tensor(X)
        yt = t.tensor(y)
        for _ in range(epochs):
            opt.zero_grad()
            loss = lossf(self.net(Xt), yt)
            loss.backward()
            opt.step()
        return self

    def predict(self, X):
        with self.torch.no_grad():
            return self.net(self.torch.tensor(X)).argmax(1).numpy()

def save_head(head: LinearHead, path):
    import torch
    torch.save({'state_dict': head.net.state_dict(), 'in_dim': head.net.in_features, 'n_classes': head.net.out_features}, path)
    return path
# --- helpers are now available as plain functions/classes in this notebook ---
print('SmartCart toolkit ready')


SmartCart toolkit ready


In [3]:
# 2) Load the cross-day bundle
# The bundle stores artifacts we create during the week: labels, indexes, weights, ONNX files.
# It is NOT the image dataset. Images are loaded in the next data-source cell.
b = open_bundle(BUNDLE_DIR)
print('bundle:', b.root)
print('artifacts:', list(b.manifest.get('artifacts', {})))


bundle: /home/jonyling/SmartCart_bundle
artifacts: ['sample_scene.jpg', 'detector.pt', 'crops_manifest.csv', 'gallery_index.npy', 'gallery_meta.csv', 'catalog_prices.csv', 'labels.csv', 'head.pt', 'per_class_metrics.csv', 'error_report.md', 'head_v2.pt', 'lift_table.csv', 'lift_table_vs_real_photo.csv', 'decisive_lift_table.csv', 'confidence_threshold.json', 'head.onnx', 'crops_train.csv', 'calibration_crops.csv']


In [4]:
# 3) Select the image data source
# SG innovation swap (D1): use our own Singapore catalog instead of the course
# default. Both return the same (tier, root) shape, so nothing downstream changes.
USE_SG_CATALOG = True
if USE_SG_CATALOG:
    tier, root = open_sg_dataset()
else:
    tier, root = open_grocery_dataset()


using data tier: sg_data (Singapore catalog)
data root: /home/jonyling/SNAIC/Week4/sg_data
OK: using real FairPrice-sourced Singapore product images.


## Find rare SKUs

**What:** Load `per_class_metrics.csv` and pick the lowest-accuracy / lowest-support classes.

**Why:** We spend our augmentation budget where the model is weakest, not uniformly.

**Watch for:** If everything is already high-acc, we still demo the pipeline on the bottom few.

In [5]:
import pandas as pd

# Classes with a genuine real natural photo (must match sg_data/compose_natural_synth.py)
NATURAL_COVERED_CLASSES = {
    'Bananas', 'Local-Mango', 'Kailan',
    'Gardenia-White-Bread', 'Pasar-Fresh-Eggs', 'Maggi-Curry-Noodles',
}

if tier == 'sg_data':
    catalog = pd.read_csv(f'{root}/catalog_prices.csv')
    studio_only = sorted(set(catalog.fine) - NATURAL_COVERED_CLASSES)
    # CLASSICAL augmentation targets all 25 classes (cheap, never measured to hurt).
    # COPY-PASTE composites are restricted separately (see the synth cell below) to
    # the 6 real-photo-covered classes - the measured sweep found all-25 composites
    # collapse real-photo accuracy from 80% to 39% via a background shortcut.
    rare = sorted(catalog.fine)
    print(f'classical augmentation for all {len(rare)} classes '
          f'({len(studio_only)} studio-only + {len(NATURAL_COVERED_CLASSES)} covered)')
else:
    pcm = b.get_table('per_class_metrics.csv')
    # Pick the weakest classes first: low accuracy, then low support.
    rare = pcm.sort_values(['acc','n']).head(3).fine.tolist()
    print('targeting rare/weak classes:', rare)


classical augmentation for all 25 classes (19 studio-only + 6 covered)


## Augment weak classes

**What:** Apply heavy photometric + geometric augmentation to existing crops of the rare classes.

**Why:** More varied views of a rare SKU teach the head invariances it couldn't learn from a few photos.

**Watch for:** An optional generative model (the Day-4 stack) could replace this - we keep a classical default so CI needs no extra deps.

In [6]:
import os, torch, torchvision.transforms.v2 as T
from PIL import Image
import numpy as np
# Start from real GroceryStoreDataset images re-listed in this runtime.
src = list_images(root)
# Classical augmentations: cheap, explainable, and deterministic enough for a lab.
aug = T.Compose([T.RandomHorizontalFlip(1.0), T.RandomRotation(25),
                 T.ColorJitter(0.4,0.4,0.4,0.1), T.RandomResizedCrop(140, scale=(0.7,1.0))])
os.makedirs('augmented', exist_ok=True)
aug_rows = []
for fine in rare:
    paths = src[src.fine==fine].path.tolist()
    if not paths:
        continue
    for k in range(8):
        p = paths[k % len(paths)]
        im = T.ToImage()(Image.open(p).convert('RGB'))
        sp = f'augmented/{fine}_{k}.jpg'
        T.ToPILImage()(aug(im)).save(sp)
        aug_rows.append({'path': sp, 'fine': fine})
aug_df = pd.DataFrame(aug_rows)
print('created', len(aug_df), 'augmented images for', len(rare), 'classes')


created 200 augmented images for 25 classes


## Copy-paste synthetic augmentation — the D4 "equivalent generation"

**What:** For each studio-only class, cut the product out of its clean FairPrice photo (GrabCut) and paste it onto a real Singapore store/market scene (`sg_data/compose_natural_synth.py`), with random scale/rotation and a feathered blend.

**Why:** This is the slide's "optional cGAN **or equivalent generation**" path. A cGAN trained from scratch on ~40 images/class tends to produce low-quality/blurry output; copy-paste is a classical, well-established technique (sometimes called "cut, paste and learn") that reliably varies the *background* the model sees — the one thing classical flip/rotate/colour-jitter augmentation can never do.

**Watch for:** These are synthetic composites, not real photos — kept in `sg_dataset/natural_synth/`, separate from the genuinely real team photos in `sg_dataset/natural/`.

In [7]:
import pathlib
# Load pre-generated copy-paste composites - but ONLY for the classes with a real
# natural photo, NOT all 25. This is a measured decision, not a guess: a 4-config
# sweep on the 17 real photos found composites-for-covered-classes-only reaches
# 80% real-photo accuracy, while composites-for-all-25-classes CRASHES it to 39%
# (worse than the 51% no-augmentation baseline). Mechanism: all 200 composites
# share the same 4 scene backgrounds, so with 25 classes pasted onto them the
# background stops identifying any class - worse, real market photos get pulled
# toward the 19 packaged-good classes that dominate the composite pool.
# To safely use the other 19 classes' composites, collect MORE distinct scene
# backgrounds first (sg_data/collect_sg_natural.py), then re-measure.
synth_df = pd.DataFrame(columns=['path', 'fine'])
if tier == 'sg_data':
    synth_root = pathlib.Path(root) / 'sg_dataset' / 'natural_synth'
    synth_rows = [{'path': str(p), 'fine': p.parent.name}
                  for p in synth_root.rglob('*.jpg')
                  if p.parent.name in NATURAL_COVERED_CLASSES]
    synth_df = pd.DataFrame(synth_rows)
print('copy-paste synthetic images (covered classes only):', len(synth_df))


copy-paste synthetic images (covered classes only): 48


## Optional: cGAN/equivalent generator

**What:** Advanced teams may generate extra rare-class images with a small conditional GAN or an equivalent generative model — **we used copy-paste synthesis above as our "equivalent generation"** since a cGAN trained on ~40 images/class tends to underperform it. This cell is left in for teams who want to try the from-scratch GAN path too; it's additive, not a replacement.

**Why:** This connects the Day-4 generative-AI material to the smart-cart problem without making it the required path.

**Watch for:** Keep `RUN_GENERATIVE_EXTENSION = False` for the baseline; if enabled, compare the lift against classical augmentation and copy-paste.

In [8]:
# Optional extension. Run the baseline first, then set this to True if your team wants the cGAN path.
RUN_GENERATIVE_EXTENSION = False
GENERATIVE_STEPS = 300
GENERATIVE_IMAGES_PER_CLASS = 8

import os, math, torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.utils import save_image

gen_df = pd.DataFrame(columns=['path', 'fine'])

if RUN_GENERATIVE_EXTENSION:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    os.makedirs('generated_cgan', exist_ok=True)
    rare_to_i = {fine: i for i, fine in enumerate(rare)}
    rare_src = src[src.fine.isin(rare)].reset_index(drop=True)

    class RareCrops(Dataset):
        def __init__(self, table):
            self.table = table
            self.tf = T.Compose([T.ToImage(), T.Resize((64,64)), T.ToDtype(torch.float32, scale=True),
                                 T.Normalize([0.5]*3, [0.5]*3)])
        def __len__(self): return len(self.table)
        def __getitem__(self, i):
            row = self.table.iloc[i]
            x = self.tf(Image.open(row.path).convert('RGB'))
            y = torch.tensor(rare_to_i[row.fine]).long()
            return x, y

    class Generator(nn.Module):
        def __init__(self, z_dim, n_classes):
            super().__init__()
            self.label = nn.Embedding(n_classes, 16)
            self.fc = nn.Linear(z_dim + 16, 256 * 8 * 8)
            self.net = nn.Sequential(
                nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(True),
                nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(True),
                nn.ConvTranspose2d(64, 3, 4, 2, 1), nn.Tanh())
        def forward(self, z, y):
            h = torch.cat([z, self.label(y)], dim=1)
            return self.net(self.fc(h).view(len(z), 256, 8, 8))

    class Discriminator(nn.Module):
        def __init__(self, n_classes):
            super().__init__()
            self.label = nn.Embedding(n_classes, 64 * 64)
            self.net = nn.Sequential(
                nn.Conv2d(4, 64, 4, 2, 1), nn.LeakyReLU(0.2, True),
                nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2, True),
                nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2, True),
                nn.Flatten(), nn.Linear(256 * 8 * 8, 1))
        def forward(self, x, y):
            label_map = self.label(y).view(len(y), 1, 64, 64)
            return self.net(torch.cat([x, label_map], dim=1)).squeeze(1)

    z_dim, n_classes = 64, len(rare)
    loader = DataLoader(RareCrops(rare_src), batch_size=16, shuffle=True, drop_last=True)
    G, D = Generator(z_dim, n_classes).to(device), Discriminator(n_classes).to(device)
    opt_g = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

    data_iter = iter(loader)
    for step in range(GENERATIVE_STEPS):
        try:
            real, y = next(data_iter)
        except StopIteration:
            data_iter = iter(loader); real, y = next(data_iter)
        real, y = real.to(device), y.to(device)
        z = torch.randn(len(real), z_dim, device=device)
        fake = G(z, y).detach()
        d_loss = F.binary_cross_entropy_with_logits(D(real, y), torch.ones(len(real), device=device)) + \
                 F.binary_cross_entropy_with_logits(D(fake, y), torch.zeros(len(real), device=device))
        opt_d.zero_grad(); d_loss.backward(); opt_d.step()

        z = torch.randn(len(real), z_dim, device=device)
        fake = G(z, y)
        g_loss = F.binary_cross_entropy_with_logits(D(fake, y), torch.ones(len(real), device=device))
        opt_g.zero_grad(); g_loss.backward(); opt_g.step()
        if (step + 1) % 100 == 0:
            print('cGAN step', step + 1, 'D', round(float(d_loss), 3), 'G', round(float(g_loss), 3))

    rows = []
    G.eval()
    with torch.no_grad():
        for fine, class_id in rare_to_i.items():
            y = torch.full((GENERATIVE_IMAGES_PER_CLASS,), class_id, device=device, dtype=torch.long)
            z = torch.randn(GENERATIVE_IMAGES_PER_CLASS, z_dim, device=device)
            imgs = G(z, y)
            for k, img in enumerate(imgs):
                path = f'generated_cgan/{fine}_{k}.jpg'
                save_image((img + 1) / 2, path)
                rows.append({'path': path, 'fine': fine})
    gen_df = pd.DataFrame(rows)
    print('created', len(gen_df), 'cGAN/equivalent generated images')
else:
    print('cGAN/equivalent extension skipped; using classical augmentation only')


cGAN/equivalent extension skipped; using classical augmentation only


## Retrain head (v2)

**What:** Rebuild features over the original images plus the selected extra images and retrain the head.

**Why:** The v2 head sees a richer long tail; classical augmentation is required, generated images are optional.

**Watch for:** Keep the class ordering identical to Day 3 so before/after numbers are comparable.

In [9]:
import torch, torchvision.transforms.v2 as T
from PIL import Image
import numpy as np
# Shared image preprocessing for DINOv2.
TF = T.Compose([T.ToImage(), T.Resize((224,224)), T.ToDtype(torch.float32, scale=True),
                T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
def feats_of(paths, model, bs=16):
    """Embed image files in small batches and return an (N, D) numpy array."""
    out=[]
    for i in range(0,len(paths),bs):
        xs=torch.stack([TF(Image.open(p).convert('RGB')) for p in paths[i:i+bs]])
        out.append(extract_features(model,[xs]))
    return np.concatenate(out,0)
from sklearn.model_selection import train_test_split
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('using device:', device)
model = load_backbone(device=device)
base = list_images(root)  # same clean basis as Day 3
classes = sorted(base.fine.unique().tolist()); cls_to_i = {c:i for i,c in enumerate(classes)}
# Hold out a clean validation split first, then add augmented images to TRAIN only.
feats_base = feats_of(list(base.path), model); yb = np.array([cls_to_i[f] for f in base.fine])
Xtr,Xval,ytr,yval,itr,ival = train_test_split(feats_base, yb, np.arange(len(yb)),
                                               test_size=0.3, random_state=0, stratify=yb)

# Hongming's crop-aware fix, carried into Day 4: Day 5 serves head_v2.pt, so if the
# base pool here were clean studio photos only, retraining would silently UNDO Day 3's
# crop-aware training. Add real detector crops to the TRAIN side of every condition.
# IMPORTANT: only Day 3's crops_train.csv split - NOT the full crops_manifest.csv.
# Day 5a calibrates the serving threshold on the held-out 30% (calibration_crops.csv);
# training head_v2 on those would make the calibration sweep read ~100% everywhere
# and pick a meaningless threshold (this actually happened - hence the split).
# NOTE: feats_base/yb stay pure studio - the decisive experiment cell below depends
# on them as its studio-only baseline pool.
INCLUDE_DETECTOR_CROPS = True
if INCLUDE_DETECTOR_CROPS:
    try:
        _crops = b.get_table('crops_train.csv')
        _crops = _crops[_crops.crop_path.apply(os.path.exists) & _crops.fine.isin(classes)]
        if len(_crops):
            fc = feats_of(list(_crops.crop_path), model)
            yc = np.array([cls_to_i[f] for f in _crops.fine])
            Xtr = np.concatenate([Xtr, fc]); ytr = np.concatenate([ytr, yc])
            print(f'+{len(_crops)} real detector crops (train split only) added to the base pool (crop-aware fix)')
    except Exception as e:
        print('no crops_train.csv in bundle (run Day 3 first) - skipping crop-aware pool:', e)

def head_with_extra(extra_df, label):
    """Fit a head on Xtr/ytr plus whatever extra images are in extra_df."""
    if len(extra_df) == 0:
        print(f'[{label}] no extra images, same as baseline')
        return LinearHead(feats_base.shape[1], len(classes)).fit(Xtr, ytr, epochs=200)
    fs = feats_of(list(extra_df.path), model)
    ys = np.array([cls_to_i[f] for f in extra_df.fine])
    Xtr_t = np.concatenate([Xtr, fs]); ytr_t = np.concatenate([ytr, ys])
    print(f'[{label}] +{len(extra_df)} training images')
    return LinearHead(feats_base.shape[1], len(classes)).fit(Xtr_t, ytr_t, epochs=200)

# Three techniques, each strictly additive over the last, so the lift table
# below can isolate copy-paste's own contribution from classical augmentation's:
#   baseline        : original studio images only
#   classical       : + flip/rotate/colour-jitter (aug_df)
#   classical+synth : + copy-paste onto real SG store/market scenes (synth_df) + optional cGAN (gen_df)
head_before    = head_with_extra(pd.DataFrame(columns=['path','fine']), 'baseline')
head_classical = head_with_extra(aug_df, 'classical')
extra_df = pd.concat([aug_df, gen_df, synth_df], ignore_index=True)
head_v2        = head_with_extra(extra_df, 'classical+copy-paste')
print('\ntotal extra training images used by classical+copy-paste:', len(extra_df),
      '(classical:', len(aug_df), 'cGAN:', len(gen_df), 'copy-paste:', len(synth_df), ')')


using device: cuda


/home/jonyling/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/jonyling/miniconda3/lib/python3.13/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


+121 real detector crops (train split only) added to the base pool (crop-aware fix)
[baseline] no extra images, same as baseline


[classical] +200 training images


[classical+copy-paste] +248 training images

total extra training images used by classical+copy-paste: 248 (classical: 200 cGAN: 0 copy-paste: 48 )


## Measure the lift

**What:** Compare before/after accuracy on the rare classes specifically and write `lift_table.csv`.

**Why:** Augmentation is only worth it if the targeted classes actually improve on held-out data.

**Watch for:** Lift can be flat on a small class subset - the pipeline and metric matter more than the number here.

In [10]:
pb = head_before.predict(Xval); pc = head_classical.predict(Xval); pa = head_v2.predict(Xval)
def acc_for(fine, pred):
    ci = cls_to_i[fine]; mask = yval==ci
    return float((pred[mask]==ci).mean()) if mask.any() else float('nan')
lift = pd.DataFrame({'fine': rare,
                     'acc_baseline':          [acc_for(f, pb) for f in rare],
                     'acc_classical':         [acc_for(f, pc) for f in rare],
                     'acc_classical_copypaste':[acc_for(f, pa) for f in rare]})
print(lift.to_string(index=False))
print('mean lift, classical only:         ', round(float((lift.acc_classical-lift.acc_baseline).mean()), 3))
print('mean lift, classical+copy-paste:   ', round(float((lift.acc_classical_copypaste-lift.acc_baseline).mean()), 3))
print("copy-paste's own marginal lift over classical:",
      round(float((lift.acc_classical_copypaste-lift.acc_classical).mean()), 3))


                    fine  acc_baseline  acc_classical  acc_classical_copypaste
     Ayam-Brand-Sardines      1.000000       1.000000                 1.000000
                 Bananas      1.000000       1.000000                 1.000000
              Camel-Nuts      0.916667       0.916667                 0.916667
                Chye-Sim      0.833333       0.750000                 0.833333
              FN-Seasons      0.916667       0.916667                 0.916667
    Gardenia-White-Bread      1.000000       1.000000                 1.000000
      Ice-Mountain-Water      0.750000       0.750000                 0.750000
        KCT-Chilli-Sauce      0.666667       0.666667                 0.666667
                  Kailan      1.000000       1.000000                 1.000000
             Kaya-Spread      1.000000       1.000000                 1.000000
     Khong-Guan-Biscuits      0.916667       0.916667                 0.916667
            Koka-Noodles      0.750000       0.91666

## Sanity check: how close does copy-paste get to a real photo?

**What:** For the 6 classes that DO have a genuine team-taken natural photo (`Kailan`, `Gardenia-White-Bread`, `Bananas`, `Pasar-Fresh-Eggs`, `Maggi-Curry-Noodles`, `Local-Mango`), compare classical+copy-paste against classical+**real** photo on the same held-out split.

**Why:** The 19 studio-only classes have no real photo to check copy-paste against — but these 6 do. If classical+copy-paste tracks classical+real reasonably closely here, that's evidence the synthetic technique is a reasonable stand-in for the 19 classes it was actually built for.

**Watch for:** This needs copy-paste composites generated *for* these 6 classes too (they're excluded by default since they don't need a synthetic stand-in) — run once with `--include-covered`:
```bash
cd Week4/sg_data && python3 compose_natural_synth.py --include-covered --classes Kailan Gardenia-White-Bread Bananas Pasar-Fresh-Eggs Maggi-Curry-Noodles Local-Mango --n-per-class 8
```

In [11]:
covered = sorted(NATURAL_COVERED_CLASSES)
if tier == 'sg_data':
    sg_root = pathlib.Path(root)
    real_rows  = [{'path': str(p), 'fine': p.parent.name}
                  for p in (sg_root/'sg_dataset'/'natural').rglob('*.jpg') if p.parent.name in covered]
    synth_cov_rows = [{'path': str(p), 'fine': p.parent.name}
                       for p in (sg_root/'sg_dataset'/'natural_synth').rglob('*.jpg') if p.parent.name in covered]
    real_df = pd.DataFrame(real_rows)
    synth_cov_df = pd.DataFrame(synth_cov_rows)
    print('real photos:', len(real_df), '| copy-paste composites for these classes:', len(synth_cov_df))

    base_cov = base[base.fine.isin(covered)]
    if len(base_cov) and len(real_df) and len(synth_cov_df):
        feats_cov = feats_of(list(base_cov.path), model)
        y_cov = np.array([cls_to_i[f] for f in base_cov.fine])
        Xtr_c, Xval_c, ytr_c, yval_c = train_test_split(feats_cov, y_cov, test_size=0.3, random_state=0, stratify=y_cov)

        def head_cov_with(extra_df):
            fs = feats_of(list(extra_df.path), model)
            ys = np.array([cls_to_i[f] for f in extra_df.fine])
            return LinearHead(feats_cov.shape[1], len(classes)).fit(
                np.concatenate([Xtr_c, fs]), np.concatenate([ytr_c, ys]), epochs=200)

        h_base   = LinearHead(feats_cov.shape[1], len(classes)).fit(Xtr_c, ytr_c, epochs=200)
        h_synth  = head_cov_with(synth_cov_df)
        h_real   = head_cov_with(real_df)

        def acc_cov(head, fine):
            ci = cls_to_i[fine]; mask = yval_c == ci
            pred = head.predict(Xval_c)
            return float((pred[mask]==ci).mean()) if mask.any() else float('nan')

        cov_table = pd.DataFrame({'fine': covered,
            'acc_baseline':    [acc_cov(h_base, f)  for f in covered],
            'acc_copypaste':   [acc_cov(h_synth, f) for f in covered],
            'acc_real_photo':  [acc_cov(h_real, f)  for f in covered]})
        print(cov_table.to_string(index=False))
        gap_closed = (cov_table.acc_copypaste - cov_table.acc_baseline).mean() / \
                     max(1e-6, (cov_table.acc_real_photo - cov_table.acc_baseline).mean())
        print(f'\ncopy-paste recovers ~{gap_closed*100:.0f}% of the lift a real photo gives, on average')
    else:
        print('run the --include-covered command above first, then re-run this cell')
else:
    print('skipped: only meaningful when USE_SG_CATALOG = True')


real photos: 17 | copy-paste composites for these classes: 48


                fine  acc_baseline  acc_copypaste  acc_real_photo
             Bananas           1.0       1.000000             1.0
Gardenia-White-Bread           1.0       0.916667             1.0
              Kailan           1.0       1.000000             1.0
         Local-Mango           1.0       1.000000             1.0
 Maggi-Curry-Noodles           1.0       1.000000             1.0
    Pasar-Fresh-Eggs           1.0       1.000000             1.0

copy-paste recovers ~-1388889% of the lift a real photo gives, on average


## Decisive experiment: real photos as the held-out TEST set, seeded training

**What:** The two measurements above are inconclusive by design: the lift table's validation split is tiny (12 images/class, so one image = 0.083 accuracy), the sanity check scores 100% on everything (studio validation images are too easy — a ceiling), and the unseeded `LinearHead` has a measured ±0.05 run-to-run noise floor (the default-dataset run trained two heads on *identical* data and they differed by 0.05). This cell fixes all three: train on **studio images only**, test on the **17 genuinely real in-store/home photos** (never seen in training, different cameras, shelf clutter, price tags), seed every head, and repeat over 5 seeds so we report **mean ± std**.

**Why:** "Does augmentation help the model survive real-world photos?" is the actual production question — a checkout camera sees messy reality, not FairPrice studio shots. Testing on real photos measures the domain gap directly, and matched budgets (+48 classical vs +48 copy-paste images, same 6 classes) make the comparison fair.

**Watch for:** n=17 test images is small — read the mean ± std and the per-image counts, not just the headline number. A lift is only real if it clears the seed-to-seed std.

In [12]:
# ---- Decisive experiment: studio-only training, real-photo test, seeded heads ----
DECISIVE_SEEDS = [0, 1, 2, 3, 4]

if tier == 'sg_data' and len(real_df) and len(synth_cov_df):
    # TEST SET: every genuinely real photo (17 images, 6 classes). Never used in training.
    feats_test = feats_of(list(real_df.path), model)
    y_test = np.array([cls_to_i[f] for f in real_df.fine])

    # TRAINING POOL: all 998 studio images across all 25 classes (feats_base/yb
    # were embedded in the retrain cell). No split needed — the test set is
    # disjoint by construction, so training uses every studio image.

    # Classical augmentations for the SAME 6 covered classes, seeded for
    # reproducibility, budget-matched to copy-paste (8 per class = 48 vs 48).
    torch.manual_seed(0); np.random.seed(0)
    os.makedirs('augmented_decisive', exist_ok=True)
    dec_rows = []
    for fine in covered:
        paths = base[base.fine == fine].path.tolist()
        for k in range(8):
            im = T.ToImage()(Image.open(paths[k % len(paths)]).convert('RGB'))
            sp = f'augmented_decisive/{fine}_{k}.jpg'
            T.ToPILImage()(aug(im)).save(sp)
            dec_rows.append({'path': sp, 'fine': fine})
    dec_aug_df = pd.DataFrame(dec_rows)

    feats_deca = feats_of(list(dec_aug_df.path), model)
    y_deca = np.array([cls_to_i[f] for f in dec_aug_df.fine])
    feats_cp = feats_of(list(synth_cov_df.path), model)
    y_cp = np.array([cls_to_i[f] for f in synth_cov_df.fine])

    conditions = {
        'baseline (studio only)': (feats_base, yb),
        '+classical (48)':        (np.concatenate([feats_base, feats_deca]), np.concatenate([yb, y_deca])),
        '+copy-paste (48)':       (np.concatenate([feats_base, feats_cp]),   np.concatenate([yb, y_cp])),
        '+both (96)':             (np.concatenate([feats_base, feats_deca, feats_cp]),
                                   np.concatenate([yb, y_deca, y_cp])),
    }

    def seeded_head(X, y, seed):
        torch.manual_seed(seed)  # full-batch fit with fixed features -> deterministic per seed
        return LinearHead(X.shape[1], len(classes)).fit(X, y, epochs=200)

    rows, per_class_hits = [], {}
    baseline_accs = None
    for name, (X, y) in conditions.items():
        accs, hits = [], np.zeros(len(real_df))
        for seed in DECISIVE_SEEDS:
            pred = seeded_head(X, y, seed).predict(feats_test)
            accs.append(float((pred == y_test).mean()))
            hits += (pred == y_test)
        accs = np.array(accs)
        if baseline_accs is None:
            baseline_accs = accs
        rows.append({'condition': name,
                     'test_acc_mean': round(float(accs.mean()), 3),
                     'test_acc_std': round(float(accs.std()), 3),
                     'mean_correct_of_17': round(float(accs.mean() * len(real_df)), 1),
                     'lift_vs_baseline': round(float((accs - baseline_accs).mean()), 3),
                     'per_seed': [round(a, 3) for a in accs]})
        per_class_hits[name] = hits / len(DECISIVE_SEEDS)

    decisive = pd.DataFrame(rows)
    print(f'TEST SET: {len(real_df)} real photos, {real_df.fine.nunique()} classes '
          f'({dict(real_df.fine.value_counts())})\n')
    print(decisive.to_string(index=False))

    # Per-class view: fraction of that class's real photos correct, averaged over seeds.
    pc = real_df[['fine']].copy()
    for name, h in per_class_hits.items():
        pc[name] = h
    per_class = pc.groupby('fine').mean().round(2)
    print('\nper-class accuracy on real photos (mean over seeds):')
    print(per_class.to_string())

    b.put_table('decisive_lift_table.csv', decisive.drop(columns=['per_seed']))
else:
    print('skipped: needs sg_data tier + real photos + covered-class composites')


TEST SET: 17 real photos, 6 classes ({'Bananas': np.int64(9), 'Pasar-Fresh-Eggs': np.int64(2), 'Local-Mango': np.int64(2), 'Maggi-Curry-Noodles': np.int64(2), 'Gardenia-White-Bread': np.int64(1), 'Kailan': np.int64(1)})

             condition  test_acc_mean  test_acc_std  mean_correct_of_17  lift_vs_baseline                            per_seed
baseline (studio only)          0.506         0.103                 8.6             0.000 [0.471, 0.353, 0.471, 0.588, 0.647]
       +classical (48)          0.494         0.060                 8.4            -0.012 [0.471, 0.412, 0.471, 0.588, 0.529]
      +copy-paste (48)          0.800         0.029                13.6             0.294 [0.824, 0.765, 0.824, 0.765, 0.824]
            +both (96)          0.788         0.047                13.4             0.282 [0.824, 0.765, 0.824, 0.706, 0.824]

per-class accuracy on real photos (mean over seeds):
                      baseline (studio only)  +classical (48)  +copy-paste (48)  +both (96)
fin

## Save + carry forward

**What:** Save head_v2 and the lift table into the bundle.

**Why:** Day 5 prefers `head_v2.pt` (this improved head) when exporting to ONNX.

**Watch for:** Confirm head_v2.pt is listed in the carry-forward print.

In [13]:
from pathlib import Path
# Final refit for SERVING: the lift-table heads above hold out 30% of the studio
# images (needed for a fair comparison), but the shipped head should learn from
# every image we have. Refit on the FULL studio pool + detector crops + extras,
# with a fixed seed (an unseeded single run measured up to ~14 points below the
# 5-seed mean on the real-photo test - seed luck, not a real difference).
torch.manual_seed(0)
fs_extra = feats_of(list(extra_df.path), model) if len(extra_df) else np.zeros((0, feats_base.shape[1]), 'float32')
ys_extra = np.array([cls_to_i[f] for f in extra_df.fine], dtype=int) if len(extra_df) else np.zeros((0,), int)
X_full = [feats_base, fs_extra]
y_full = [yb, ys_extra]
try:
    _crops = b.get_table('crops_train.csv')
    _crops = _crops[_crops.crop_path.apply(os.path.exists) & _crops.fine.isin(classes)]
    if len(_crops):
        X_full.append(feats_of(list(_crops.crop_path), model))
        y_full.append(np.array([cls_to_i[f] for f in _crops.fine]))
except Exception:
    pass
head_serve = LinearHead(feats_base.shape[1], len(classes)).fit(
    np.concatenate(X_full), np.concatenate(y_full), epochs=200)
print(f'serving head refit on {sum(len(y) for y in y_full)} images (full studio + crops + extras), seed 0')

save_head(head_serve, str(Path(b.root)/'head_v2.pt')); b._note('head_v2.pt')
b.put_table('lift_table.csv', lift)
if tier == 'sg_data' and 'cov_table' in dir():
    b.put_table('lift_table_vs_real_photo.csv', cov_table)
save_bundle(b)
print('▶ Carries forward to Day 5: head_v2.pt (full-pool refit) + lift_table.csv')


serving head refit on 1367 images (full studio + crops + extras), seed 0
[bundle] saved -> /home/jonyling/SmartCart_bundle
▶ Carries forward to Day 5: head_v2.pt (full-pool refit) + lift_table.csv
